<a href="https://colab.research.google.com/github/peterbabulik/QuantumWalker/blob/main/QGF_QRN_dieharder_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install qiskit qiskit-aer cma

ERROR: Could not find a version that satisfies the requirement dieharder (from versions: none)
ERROR: No matching distribution found for dieharder


In [6]:
!sudo apt-get update && sudo apt-get install -y dieharder

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,566 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [3,404 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Package

In [7]:
# In a new Colab cell
!dieharder -V

3.31.1


In [4]:

# ==============================================================================
#  AI-DRIVEN QUANTUM RANDOMNESS FORGE (Final Corrected Version)
# ==============================================================================
import numpy as np
import time
import os
import subprocess
import warnings
from scipy.linalg import expm
from qiskit import transpile

# Qiskit Imports
import qiskit
from qiskit.circuit import QuantumCircuit, Gate
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.synthesis.two_qubit import TwoQubitBasisDecomposer
from qiskit.circuit.library import CXGate
from qiskit_aer import AerSimulator

# The powerful "Designer AI" engine
try:
    import cma
    cma_available = True
except ImportError:
    cma_available = False

# Suppress benign warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

print(f"Qiskit version: {qiskit.__version__}")
if not cma_available:
    print("\nCRITICAL ERROR: 'cma' library not found. Please run: pip install cma")
    exit()

# --- CORE FORGE AND QRNG FUNCTIONS ---
PAULI_BASIS_2Q = [p1+p2 for p1 in 'IXYZ' for p2 in 'IXYZ' if p1+p2 != 'II']
decomposer = TwoQubitBasisDecomposer(CXGate())
IDEAL_PROBS_2Q = np.array([0.25, 0.25, 0.25, 0.25])

def build_gate_from_coeffs(coeffs: np.ndarray) -> Gate:
    generator_h = SparsePauliOp(PAULI_BASIS_2Q, coeffs=coeffs)
    u_matrix = expm(-1j * generator_h.to_matrix())
    forged_gate = Gate(name="AI_ENTROPY", num_qubits=2, params=[])
    forged_gate.definition = decomposer(u_matrix)
    return forged_gate

def randomness_fitness_function(coeffs: np.ndarray) -> float:
    try:
        gate = build_gate_from_coeffs(coeffs)
        qc = QuantumCircuit(2)
        qc.append(gate, [0, 1])
        sv = Statevector.from_int(0, 4).evolve(qc)
        real_probs = sv.probabilities()
        return np.sum((real_probs - IDEAL_PROBS_2Q)**2)
    except Exception:
        return 1.0

if __name__ == "__main__":
    # ==============================================================================
    #  ACT 1: THE FORGE - AI DESIGNS THE ENTROPY GATE
    # ==============================================================================
    print("\n--- Act 1: The Forge - AI Searching for the Ultimate Entropy Gate ---")
    es = cma.CMAEvolutionStrategy(np.random.rand(15) * 2 * np.pi - np.pi, 0.5, {'bounds': [-np.pi, np.pi], 'maxfevals': 1000, 'verbose': -9})
    print("  > AI is thinking... (This may take a minute)")
    es.optimize(randomness_fitness_function)
    print("  > AI has found a solution.")
    best_fitness = es.result.fbest
    best_coeffs = es.result.xbest
    print(f"\n  > Champion Gate found with fitness score (lower is better): {best_fitness:.8f}")
    CHAMPION_ENTROPY_GATE = build_gate_from_coeffs(best_coeffs)
    print("\n--- AI-Forged 'Entropy Gate' Algorithm ---")
    print("The discovered QRNG algorithm is the following 2-qubit gate:")
    print(CHAMPION_ENTROPY_GATE.definition.draw(output='text'))

    # ==============================================================================
    #  ACT 2: THE HARVEST - GENERATING 1MB OF QUANTUM DATA
    # ==============================================================================
    print("\n--- Act 2: The Harvest - Generating 1MB of Random Data ---")
    FILENAME = "ai_quantum_random.bin"
    TARGET_BYTES = 1 * 1024 * 1024
    BATCH_SIZE_SHOTS = 8192
    BITS_PER_SHOT = 2
    qrng_circuit = QuantumCircuit(BITS_PER_SHOT, BITS_PER_SHOT)
    qrng_circuit.append(CHAMPION_ENTROPY_GATE, [0, 1])
    qrng_circuit.measure_all()
    simulator = AerSimulator()
    bytes_written = 0
    start_time = time.time()
    print(f"  > Target file: '{FILENAME}'")

    print("  > Transpiling AI circuit for the AerSimulator...")
    transpiled_circuit = transpile(qrng_circuit, simulator)

    print(f"  > Generating {TARGET_BYTES:,} bytes of data...")
    with open(FILENAME, "wb") as f:
        while bytes_written < TARGET_BYTES:
            job = simulator.run(transpiled_circuit, shots=BATCH_SIZE_SHOTS, memory=True)
            results = job.result()
            memory = results.get_memory()

            # --- FIX IS HERE: Sanitize the string by removing spaces ---
            byte_array = bytearray(int(bitstring.replace(' ', ''), 2) for bitstring in memory)
            # --- END OF FIX ---

            f.write(byte_array)
            bytes_written += len(byte_array)
            progress = (bytes_written / TARGET_BYTES) * 100
            print(f"\r  > Progress: {progress:6.2f}% ({bytes_written:,}/{TARGET_BYTES:,} bytes)", end="")
    end_time = time.time()
    print(f"\n\n  > Successfully wrote {bytes_written:,} bytes to '{FILENAME}' in {end_time - start_time:.2f} seconds.")

    # ==============================================================================
    #  ACT 3: THE INQUISITION - ANALYSIS WITH DIEHARDER
    # ==============================================================================
    print("\n--- Act 3: The Inquisition - Statistical Analysis with Dieharder ---")
    try:
        result = subprocess.run(["dieharder", "-V"], capture_output=True, text=True, check=True)
        print(f"  > Found Dieharder: {result.stdout.strip()}")
        dieharder_found = True
    except (subprocess.CalledProcessError, FileNotFoundError):
        print("\n  CRITICAL WARNING: 'dieharder' command not found.")
        print("  Please install it to perform the final analysis.")
        dieharder_found = False

    if dieharder_found:
        print(f"\n  > Running all Dieharder tests on '{FILENAME}'. This will take several minutes...")
        try:
            command = ["dieharder", "-a", "-f", FILENAME]
            process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            while True:
                output = process.stdout.readline()
                if output == '' and process.poll() is not None:
                    break
                if output:
                    print(output.strip())
            stderr_output = process.stderr.read()
            if process.returncode != 0:
                print("\n--- DIEHARDER ERROR ---")
                print(stderr_output)
            else:
                print("\n--- ANALYSIS COMPLETE ---")
                print("Review the results above. Each test provides a 'p-value'.")
                print("A 'PASSED' assessment is excellent. A 'FAILED' assessment indicates a potential flaw.")
        except Exception as e:
            print(f"\n  An error occurred while running Dieharder: {e}")

Qiskit version: 2.1.0

--- Act 1: The Forge - AI Searching for the Ultimate Entropy Gate ---
  > AI is thinking... (This may take a minute)
  > AI has found a solution.

  > Champion Gate found with fitness score (lower is better): 0.00000075

--- AI-Forged 'Entropy Gate' Algorithm ---
The discovered QRNG algorithm is the following 2-qubit gate:
global phase: 3.7141
     ┌─────────────────────────┐        ┌─────────────────────┐        »
q_0: ┤ U(0.83079,2.9103,3.038) ├───■────┤ U(1.1163,-π/2,-π/2) ├─────■──»
     ├─────────────────────────┴┐┌─┴─┐┌─┴─────────────────────┴──┐┌─┴─┐»
q_1: ┤ U(1.0594,1.892,-0.98335) ├┤ X ├┤ U(1.9623,2.7283,0.71595) ├┤ X ├»
     └──────────────────────────┘└───┘└──────────────────────────┘└───┘»
«         ┌────────────────────┐        ┌────────────────────────────┐
«q_0: ────┤ U(0.021611,-π,π/2) ├─────■──┤ U(0.61583,2.0672,-0.97646) ├
«     ┌───┴────────────────────┴──┐┌─┴─┐├───────────────────────────┬┘
«q_1: ┤ U(1.1033,0.32306,-2.2097) ├┤ X ├┤ U(1.4648,-1

In [8]:

# The -a flag runs all tests.
# The -f flag specifies the file to test.
!dieharder -a -f ai_quantum_random.bin

#=============================================================================#
#            dieharder version 3.31.1 Copyright 2003 Robert G. Brown          #
#=============================================================================#
   rng_name    |           filename             |rands/second|
        mt19937|           ai_quantum_random.bin|  6.14e+07  |
#=============================================================================#
        test_name   |ntup| tsamples |psamples|  p-value |Assessment
#=============================================================================#
   diehard_birthdays|   0|       100|     100|0.88608347|  PASSED  
      diehard_operm5|   0|   1000000|     100|0.80355954|  PASSED  
  diehard_rank_32x32|   0|     40000|     100|0.09260010|  PASSED  
    diehard_rank_6x8|   0|    100000|     100|0.27878019|  PASSED  
   diehard_bitstream|   0|   2097152|     100|0.86389365|  PASSED  
        diehard_opso|   0|   2097152|     100|0.31103631|  PASSED 